In [1]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import lib.dataloader as dl
import lib.datasplit as ds
import lib.metrics as me
import encoding.kmer_freq as kf


# Data Processing
import pandas as pd
import numpy as np

# Modelling
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, train_test_split, GridSearchCV
from scipy.stats import randint

# Tree Visualisation
from sklearn.tree import export_graphviz
from IPython.display import Image
import graphviz

In [2]:
from immuneML.encodings.kmer_frequency import KmerFreqRepertoireEncoder
from immuneML.data_model.datasets import RepertoireDataset

In [3]:
from immuneML.dsl.ImmuneMLParser import ImmuneMLParser

yaml_path = Path("/home/ubuntu/Project_7/rebecca_folder/cmsb26_project7/models/my.yaml")
result_path = Path("/home/ubuntu/Project_7/rebecca_folder/cmsb26_project7/models/results")

symbol_table, specification_path = ImmuneMLParser.parse_yaml_file(yaml_path, result_path)

/vol/data/conda_envs/rand_forest_env/lib/python3.10/site-packages/airr/interface.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/vol/data/conda_envs/rand_forest_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-28 22:05:37.407162: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769637937.426162 3984614 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when 

2026-01-28 22:06:22.923502: Imported repertoire dataset my_dataset:
- Example count: 200
- Labels: {'disease_signal', 'sim_item', 'disease', 'control_signal'}
2026-01-28 22:06:22.955606: Full specification is available at /home/ubuntu/Project_7/rebecca_folder/cmsb26_project7/models/results/full_my.yaml.



In [ ]:
specification_path # symbole_table: SymbolTable()

PosixPath('/home/ubuntu/Project_7/rebecca_folder/cmsb26_project7/models/results/full_my.yaml')

In [7]:
from immuneML.dsl.symbol_table.SymbolType import SymbolType
from immuneML.dsl.semantic_model.SemanticModel import SemanticModel

instructions = symbol_table.get_by_type(SymbolType.INSTRUCTION)
output = symbol_table.get("output")
model = SemanticModel([instruction.item for instruction in instructions], result_path, output)

In [13]:
model.instructions[0].__dict__

{'state': TrainMLModelState(dataset=RepertoireDataset(identifier='f9c0cc93de614b4f89128f818159f381', name='my_dataset', encoded_data=None, labels={'control_signal': [True], 'disease_signal': [True], 'sim_item': ['sim_item_healthy', 'sim_item_disease'], 'disease': [False, True], 'organism': None}, dataset_file=PosixPath('/home/ubuntu/Project_7/rebecca_folder/cmsb26_project7/models/results/datasets/my_dataset/my_dataset.yaml')), hp_strategy=<immuneML.hyperparameter_optimization.strategy.GridSearch.GridSearch object at 0x76c0fb7485e0>, hp_settings=[<immuneML.hyperparameter_optimization.HPSetting.HPSetting object at 0x76c0fb748730>], assessment=<immuneML.hyperparameter_optimization.config.SplitConfig.SplitConfig object at 0x76c0fb748ac0>, selection=<immuneML.hyperparameter_optimization.config.SplitConfig.SplitConfig object at 0x76c0fb749c00>, metrics={<ClassificationMetric.AUC: 'roc_auc_score'>}, optimization_metric=<ClassificationMetric.BALANCED_ACCURACY: 'balanced_accuracy_score'>, label

In [14]:
from immuneML.hyperparameter_optimization.core.HPAssessment import HPAssessment

my_instruction = model.instructions[0]
my_instruction.state.path = model.result_path
my_instruction.state = HPAssessment.run_assessment(my_instruction.state)

2026-01-28 22:17:35.722209: Training ML model: running outer CV loop: started split 1/1.

2026-01-28 22:17:35.727332: Hyperparameter optimization: running the inner loop of nested CV: selection for label disease (label 1 / 1).

2026-01-28 22:17:35.728532: Evaluating hyperparameter setting: my_kmer_encoding_my_random_forest...
2026-01-28 22:17:35.729355: Encoding started...
2026-01-28 22:17:51.891547: Encoding finished.
2026-01-28 22:17:51.893360: ML model training started...
2026-01-28 22:17:52.856307: ML model training finished.
2026-01-28 22:17:52.857278: Encoding started...
2026-01-28 22:18:00.621097: Encoding finished.
2026-01-28 22:18:00.686164: Completed hyperparameter setting my_kmer_encoding_my_random_forest.

2026-01-28 22:18:00.687504: Hyperparameter optimization: running the inner loop of nested CV: completed selection for label disease (label 1 / 1).

2026-01-28 22:18:00.688220: Training ML model: running the inner loop of nested CV: retrain models for label disease (label 

In [15]:
from immuneML.hyperparameter_optimization.core.HPUtil import HPUtil

state = HPAssessment._create_root_path(my_instruction.state)
train_val_datasets, test_datasets = HPUtil.split_data(state.dataset, state.assessment, state.path, state.label_configuration)

n_splits = len(train_val_datasets)

#for index in range(n_splits):
    #state = HPAssessment.run_assessment_split(state, train_val_datasets[index], test_datasets[index], index, n_splits)

In [16]:
test_datasets

[RepertoireDataset(identifier='0f5dc5cd-72d8-4924-b1ba-9aba19124eca', name='0f5dc5cd-72d8-4924-b1ba-9aba19124eca', encoded_data=None, labels={'control_signal': [True], 'disease_signal': [True], 'sim_item': ['sim_item_healthy', 'sim_item_disease'], 'disease': [False, True], 'organism': None}, dataset_file=None)]

In [18]:
from immuneML.data_model.datasets.RepertoireDataset import RepertoireDataset

metadata_file = Path("/vol/data/immuneML/simulation_data/data/repertoire_LigoSim_200_balanced/my_sim_inst/exported_dataset/airr/simulated_dataset.csv")

my_rep_dataset = RepertoireDataset.build(
    metadata_file=metadata_file,
    name="200_balanced",
    repertoire_ids=["rep1","rep2"],
    metadata_fields=["subject_id","disease_status"]
)

my_rep_dataset

FileNotFoundError: [Errno 2] No such file or directory: '/vol/data/immuneML/simulation_data/data/repertoire_LigoSim_200_balanced/my_sim_inst/exported_dataset/airr/4e5820fee49e43e39abd043e14f0899d_metadata.yaml'

In [44]:
train_val_dataset, test_dataset, split_index, n_splits = train_val_datasets[0], test_datasets[0], 0, n_splits

from immuneML.hyperparameter_optimization.core.HPSelection import HPSelection
from immuneML.hyperparameter_optimization.core.HPAssessment import HPAssessmentState

current_path = HPAssessment.create_assessment_path(state, split_index)

assessment_state = HPAssessmentState(split_index, train_val_dataset, test_dataset, current_path,
                                        state.label_configuration)
state.assessment_states.append(assessment_state)

#state = HPSelection.run_selection(state, train_val_dataset, current_path, split_index)